In [3]:
# imports

import os
import requests
from dotenv import load_dotenv;
from bs4 import BeautifulSoup
from IPython.display import Markdown, display;
from openai import OpenAI;

In [4]:
# Load environment variables in a file called .env
load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

# Check the api_key
if not api_key:
  print("No API key")
else:
  print("API key found")

API key found


In [5]:
openai = OpenAI()


In [6]:
# Some websites need you to yse proper headers when scraping
headers = {
  "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

# A class to represent a Webpage
class Webpage:
  """
  A utility class to represent a Website that we have scraped
  """

  def __init__(self, url):
    """
    Create this Website object from the given url using the BeautifulSoup library
    """
    self.url = url
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    self.title = soup.title.string if soup.title else "No title found"
    for irrelevant in soup.body(["script", "style", "img", "input"]):
      irrelevant.decompose()
    self.text = soup.body.get_text(separator="\n", strip=True)


In [7]:
ed = Webpage("https://edwarddonner.com")
print(ed.title)
print(ed.text)

Home - Edward Donner
Home
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy DJing (but I’m badly out of practice), amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. Recruiters use our product today to source, understand, engage and manage talent. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
We work with groundbreaking, proprietary LLMs verticalized for talent, we’ve
patented
our matching model, and our award-winning platform has happy customers and tons of press coverage.
Connec

In [8]:
# Define the system prompt ex: change to respond in spanish etc.
system_prompt = "You are an assistant that analyzes the contents of a website \
  and provides a short summary, ignoring text that might be navigation related. \
  Respond in markdown."

In [ ]:
# A function that writes a user prompt that asks for summaries of websites
def user_prompt_for(website):
  user_prompt = f"You are looking at a website titled {website.title} \n"
  user_prompt += "The contents of this website is as follows; \
    please provide a short summary of this website in markdown. \
    If it includes news or announcements, then summarize these too.\n\n"
  user_prompt += website.text
  return user_prompt

In [14]:
print(user_prompt_for(ed))

You are looking at a website titled Home - Edward Donner 
The contents of this website is as follows;     please provide a short summary of this website in markdown.     If it includes news or announcements, then summarize these too.

Home
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy DJing (but I’m badly out of practice), amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. Recruiters use our product today to source, understand, engage and manage talent. I’m previously the founder and CEO of AI startup unta

In [15]:
# define the message
def messages_for(website):
  return [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt_for(website)}
  ]

In [ ]:
# call the OpenAI API

def summarize(url):
  website = Webpage(url)
  response = openai.chat.completions.create(
    model = "gpt-4o-mini",
    messages = messages_for(website)
  )
  return response.choices[0].message.content


In [17]:
summarize("https://edwarddonner.com")

'# Website Summary - Edward Donner\n\nThe website is a personal platform for Ed, who is passionate about coding and experimenting with large language models (LLMs). He is the co-founder and CTO of Nebula.io, which focuses on leveraging AI to assist individuals in discovering their potential and engaging with talent management. Ed has a background as the founder and CEO of the AI startup untapt, acquired in 2021, and emphasizes his work with patented matching models and proprietary LLMs.\n\n## Key Offerings:\n- **Connect Four & Outsmart**: These features highlight environments where LLMs engage in strategic contests, emphasizing diplomacy and cunning.\n  \n## Recent Announcements:\n1. **May 28, 2025**: Launch of courses aimed at cultivating expertise in LLM technology.\n2. **May 18, 2025**: Announcement of a 2025 AI Executive Briefing.\n3. **April 21, 2025**: Introduction of "The Complete Agentic AI Engineering Course."\n4. **January 23, 2025**: Workshop announcement for hands-on experi

In [18]:
# function to make it pretty
def display_summary(url):
  summary = summarize(url)
  display(Markdown(summary))

In [19]:
display_summary("https://edwarddonner.com")

# Summary of Edward Donner's Website

Edward Donner's website serves as a personal space for sharing his interests and professional endeavors, particularly in the field of AI and LLMs (Large Language Models). Ed describes himself as a code enthusiast and a hobbyist in DJing and electronic music production. He is the co-founder and CTO of Nebula.io, a company focused on leveraging AI for talent discovery and management, and has a background as the founder of the AI startup untapt, which was acquired in 2021.

## Key Highlights:
- **Professional Background**: Ed runs Nebula.io, which uses proprietary LLMs for talent engagement and management.
- **Interests**: Coding, LLM experimentation, DJing, and electronic music.
- **Recent Courses and Announcements**:
  - **May 28, 2025**: Courses aimed at becoming an LLM expert and leader.
  - **May 18, 2025**: Upcoming 2025 AI Executive Briefing.
  - **April 21, 2025**: Launch of "The Complete Agentic AI Engineering Course."
  - **January 23, 2025**: Resources for a hands-on LLM workshop on agents.

Overall, the site reflects Ed's passion for AI and his commitment to educating others in the field.

In [20]:
display_summary("https://cnn.com")

# CNN Website Summary

CNN is a comprehensive news website that covers a wide range of topics, including current events, politics, business, health, science, and entertainment. The platform provides live broadcasts, videos, and articles, allowing users to stay informed on breaking news and global developments.

## Key News Highlights

- **Israel-Hamas Ceasefire Proposal**: Israel has accepted a new ceasefire proposal from the US amid ongoing military operations in Gaza, despite civilian hardships.
- **Ukraine-Russia Conflict**: Recent meetings in Kyiv with US senators cast doubt on the upcoming peace talks between Ukraine and Russia.
- **Natural Disasters**: At least 150 fatalities were reported due to deadly floods in Nigeria.
- **Cultural Loss**: Loretta Swit, known for her role in the series "M.A.S.H.", has passed away.

In addition to news, CNN also features various interactive content, such as games and newsletters, alongside investigative pieces and profiles on significant events and figures. The website emphasizes user engagement by inviting feedback on advertising and user experience.

In [21]:
display_summary("https://lankacnews.com/")

# Summary of "Attention Required! | Cloudflare"

The website you attempted to access, **lankacnews.com**, has blocked your request due to security measures provided by Cloudflare. The block may have been triggered by specific actions such as providing certain words, phrases, or malformed data.

### Key Points:
- **Block Reason**: Security protection against potential online threats.
- **Resolution**: You can contact the website owner, providing details about your action that led to the block, along with the Cloudflare Ray ID for reference.
- **Technical Details**: Includes your IP address and a unique Cloudflare Ray ID for troubleshooting.

No news or announcements are included in this message.